# Module 18 — Testing, Debugging, and Quality

## Exercise 18.3 — Six real bugs, findable by property testing

Every function below has a bug that example-based tests miss and a property
test finds in seconds. Write the property, watch it fail, read the shrunk
counterexample, then fix the function.
Run:  pip install hypothesis && pytest ex03_hypothesis.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. Fixtures

In [ ]:
@pytest.fixture
def db() -> Iterator[Database]:
    conn = Database(":memory:")
    conn.migrate()
    yield conn              # the test runs here
    conn.close()            # teardown, even if the test fails

def test_insert(db: Database) -> None:
    db.insert({"id": 1})
    assert db.count() == 1

A fixture is dependency injection for tests (Module 12), and it composes:
fixtures can request other fixtures.

```text
@pytest.fixture(scope="session")     # once per test run
@pytest.fixture(scope="module")      # once per test file
@pytest.fixture(scope="function")    # the default: once per test
```


**Scope is a trade between speed and isolation, and getting it wrong is the
most common cause of "passes alone, fails in the suite".** A session-scoped
fixture holding mutable state is shared by every test; one test mutating it
changes what the others see, and the failure depends on *ordering*. Session
scope is for genuinely immutable or externally-managed things: a started
container, a compiled asset, a read-only fixture file.

### The built-in fixtures worth knowing

In [ ]:
def test_files(tmp_path: Path) -> None:          # a fresh temp dir per test
    (tmp_path / "f.txt").write_text("x", encoding="utf-8")

def test_env(monkeypatch: pytest.MonkeyPatch) -> None:
    monkeypatch.setenv("API_KEY", "test")        # undone automatically
    monkeypatch.setattr(module, "CONSTANT", 42)
    monkeypatch.chdir(tmp_path)

def test_output(capsys: pytest.CaptureFixture[str]) -> None:
    run()
    assert "done" in capsys.readouterr().out

def test_logs(caplog: pytest.LogCaptureFixture) -> None:
    with caplog.at_level(logging.WARNING):
        risky()
    assert "retrying" in caplog.text

`monkeypatch` undoes everything at teardown, which `setattr` by hand does not.

### `conftest.py`

Fixtures defined there are available to every test in that directory and below,
with no import. Put shared fixtures there; put nothing else there, because code
in `conftest.py` is invisible to a reader of the test file.

---

## Concept 4. Test doubles, and why fakes beat mocks

| Kind | What it is | Use when |
|---|---|---|
| **Dummy** | A placeholder never used | Filling a required parameter |
| **Stub** | Returns canned answers | You need a specific return value |
| **Fake** | A working, simpler implementation | **The default choice** |
| **Spy** | Records how it was called | You must assert on an interaction |
| **Mock** | A spy with pre-set expectations | Rarely |

In [ ]:
# a FAKE: a real implementation, in memory
class InMemoryUserRepo:
    def __init__(self) -> None:
        self._users: dict[int, User] = {}
    def save(self, user: User) -> None:
        self._users[user.id] = user
    def get(self, uid: int) -> User | None:
        return self._users.get(uid)

def test_registration_stores_the_user() -> None:
    repo = InMemoryUserRepo()
    service = RegistrationService(repo)
    service.register("ada@example.com")
    assert repo.get(1).email == "ada@example.com"     # asserts an OUTCOME

versus

In [ ]:
def test_registration_calls_save() -> None:
    repo = Mock()
    RegistrationService(repo).register("ada@example.com")
    repo.save.assert_called_once()                     # asserts an INTERACTION

The mock version passes if `save` is called **with the wrong user**. It also
breaks the moment you rename `save` or call it twice for a good reason. A mock
test is coupled to the implementation; a fake test is coupled to the behaviour.

**Use a mock only when the interaction *is* the requirement** — "an email was
sent", "the audit log recorded it", "the payment gateway was called exactly
once".

### Patching: where, not what

In [ ]:
# app/service.py
from app.clients import fetch_user      # a COPY of the reference (Module 06)

# test
patch("app.clients.fetch_user")         # WRONG: service.py's copy is unaffected
patch("app.service.fetch_user")         # RIGHT: patches the name in use

**Patch where the name is used, not where it is defined.** This is Module 06's
`from x import y` binding rule, and it accounts for a large share of "the patch
did nothing" confusion.

Better still: do not patch. If the dependency is injected (Module 12), you pass
a fake and no patching is needed. **Heavy patching is a design smell** — it is
usually telling you the code constructs its own dependencies.

---

## Concept 5. Coverage, and what it does not tell you

```bash
pytest --cov=src --cov-report=term-missing
pytest --cov=src --cov-branch          # branch coverage: much more honest
```

Coverage tells you which lines *ran*. It does not tell you whether anything was
*asserted*:

In [ ]:
def test_nothing() -> None:
    process_everything()        # 100% coverage, zero assertions, always passes

Use coverage to **find untested code**, never as a quality target. A number
target produces tests written to raise the number, which are worse than no
tests because they take time to run and give false confidence.

Branch coverage is worth enabling: a line with `if x:` counts as covered when
only the true branch ever ran.

**Where to look in a coverage report:** error-handling paths (usually the least
covered and the most dangerous), boundary conditions, and any file with high
coverage and few assertions.

---

## Concept 6. Property-based testing

In [ ]:
from hypothesis import given, strategies as st

@given(st.lists(st.integers()))
def test_sort_is_idempotent(items: list[int]) -> None:
    assert sorted(sorted(items)) == sorted(items)

@given(st.text())
def test_encode_decode_round_trip(s: str) -> None:
    assert s.encode("utf-8").decode("utf-8") == s

@given(st.decimals(min_value=0, max_value=10**6, places=2), st.integers(1, 100))
def test_money_allocation_is_exact(amount: Decimal, n: int) -> None:
    parts = Money.parse(str(amount)).allocate(n)
    assert sum(parts[1:], parts[0]) == Money.parse(str(amount))

You state a **property** and Hypothesis searches for a counterexample, then
**shrinks** it to the smallest failing input. That shrinking is the feature: it
turns "fails on a 400-element list" into "fails on `[0, 0]`".

Properties worth reaching for: round trips (encode/decode, serialise/parse),
invariants (a sort's output is a permutation of its input; a total is
conserved), idempotence, commutativity, and comparison against a slow-but-obvious
implementation.

Property tests find the inputs you would never think to write: empty, one
element, duplicates, `NaN`, surrogate pairs, a 10,000-element list, `-0.0`.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: pytest, the parts you need
- Section 2: Fixtures
- Section 3: `parametrize`
- Section 4: Test doubles, and why fakes beat mocks
- Section 5: Coverage, and what it does not tell you
- Section 6: Property-based testing
- Section 7: Debugging
- Section 8: Linting and formatting
- Section 9: Designing for testability

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import math
from decimal import Decimal
from typing import Any

from hypothesis import given
from hypothesis import strategies as st


# --- 1 ------------------------------------------------------------------------

---

## `normalise_whitespace`

Collapse runs of whitespace to single spaces and trim.

In [ ]:
def normalise_whitespace(text: str) -> str:
    """Collapse runs of whitespace to single spaces and trim."""
    return " ".join(text.split(" ")).strip()

---

## `chunk`

Split into chunks of at most `size`.

In [ ]:
def chunk(items: list[Any], size: int) -> list[list[Any]]:
    """Split into chunks of at most `size`."""
    return [items[i:i + size] for i in range(0, len(items), size)]

---

## `percentage`

part as a percentage of whole, rounded to one decimal place.

In [ ]:
def percentage(part: float, whole: float) -> float:
    """part as a percentage of whole, rounded to one decimal place."""
    return round(part / whole * 100, 1)

---

## `merge_sorted`

Merge two sorted lists into one sorted list.

In [ ]:
def merge_sorted(a: list[int], b: list[int]) -> list[int]:
    """Merge two sorted lists into one sorted list."""
    result = []
    i = j = 0
    while i < len(a) and j < len(b):
        if a[i] < b[j]:
            result.append(a[i]); i += 1
        else:
            result.append(b[j]); j += 1
    result.extend(a[i:])
    return result

---

## `truncate`

Truncate to `limit` characters, appending '...' if it was cut.

In [ ]:
def truncate(text: str, limit: int) -> str:
    """Truncate to `limit` characters, appending '...' if it was cut."""
    if len(text) <= limit:
        return text
    return text[:limit - 3] + "..."

---

## `split_evenly`

Split `total` into `parts` whole numbers summing to total.

In [ ]:
def split_evenly(total: int, parts: int) -> list[int]:
    """Split `total` into `parts` whole numbers summing to total."""
    base = total // parts
    result = [base] * parts
    result[0] += total - base * parts
    return result

---

## `test_normalise_whitespace`

_test normalise whitespace_

In [ ]:
@given(st.text())
def test_normalise_whitespace(text: str) -> None:
    result = normalise_whitespace(text)
    assert "\t" not in result
    assert "\n" not in result
    assert "  " not in result

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
# TODO: write the other five.


ANSWERS = """
Record the shrunk counterexample for each:

1  normalise_whitespace :
2  chunk                :
3  percentage           :
4  merge_sorted         :
5  truncate             :
6  split_evenly         :

Then answer:
- which of the six would you have found with example-based tests?
- which counterexample would you never have thought to write by hand?
- for bug 4, what does the SECOND property (permutation) catch that the first
  (sorted) does not? Construct an implementation that passes one and fails the
  other.
"""

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.